# AI-Driven Phishing Email Detection

This notebook demonstrates the complete workflow for detecting phishing emails, from dataset integrity auditing to feature engineering, baseline model comparison, and out-of-distribution manual validation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')

## 1. Dataset Loading and Overview

In [ ]:
df = pd.read_csv('../data/raw/Zero-Day_Phishing_Emails_Corpus.csv')
df.head()

In [ ]:
print(f'Total records: {len(df)}')
print(df['Label'].value_counts(normalize=True))

## 2. Dataset Integrity Checks
Our provenance analysis indicated potential synthetic generation or template reuse. Let's quantify this.

In [ ]:
from src.data.dataset_audit import audit_duplicates
audit_duplicates(df)

## 3. Feature Engineering
We extract structural markers (URLs, HTML, Punctuation, Urgency) prior to vectorizing the text.

In [ ]:
from src.features.build_features import (
    URLFeatureExtractor, HTMLFeatureExtractor,
    PunctuationFeatureExtractor, UrgencyFeatureExtractor
)

# Example of URL extraction on a small subset
subset = df.sample(5, random_state=42)
url_ext = URLFeatureExtractor()
url_features = url_ext.fit_transform(subset)
url_features

## 4. Train/Test Methodology and Model Training
Because ~67.76% of the dataset shares subject templates, we use a `GroupShuffleSplit` (group-aware split) on normalized subjects to prevent data leakage across the train/test boundary.

In [ ]:
from src.models.train import build_training_pipeline
from sklearn.model_selection import GroupShuffleSplit
import re

# Creating normalized subjects for grouping
def normalize_subject(subj):
    s = str(subj).lower().strip()
    return re.sub(r'[^a-z0-9]', '', s)

df['Norm_Subject'] = df['Subject'].apply(normalize_subject)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df['Norm_Subject']))

X_train, y_train = df.iloc[train_idx][['Subject', 'Body']], df.iloc[train_idx]['Label']
X_test, y_test = df.iloc[test_idx][['Subject', 'Body']], df.iloc[test_idx]['Label']

print(f'Training set size: {len(X_train)}')
print(f'Test set size: {len(X_test)}')

Let's train the selected deployment model: Logistic Regression.

In [ ]:
pipe = build_training_pipeline('logistic_regression', stop_words=None)
pipe.fit(X_train, y_train)

## 5. Evaluation Metrics and Confusion Matrix
Let's evaluate the model on the held-out test set.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

preds = pipe.predict(X_test)
print(classification_report(y_test, preds))

cm = confusion_matrix(y_test, preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Legitimate', 'Phishing'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix on Test Set')
plt.show()

## 6. Artifact Analysis
Perfect confusion matrices across multiple model families prompted further investigation, because such uniformly perfect performance is unusual for realistic phishing-email classification. Let's look at the distribution of real URLs.

In [ ]:
all_url_features = url_ext.fit_transform(df)
url_analysis = pd.concat([df['Label'], all_url_features], axis=1)
url_analysis.groupby('Label')[['num_real_urls', 'num_masked_urls']].mean()

> **Finding:** No standard real URLs matching the implemented extraction pattern were detected in the legitimate class of the analyzed dataset. The model learned to use the presence of a URL as a near-perfect shortcut feature for Phishing.

## 7. Manual External Validation
To assess true generalizability, an illustrative manual external validation set of representative examples was evaluated through the serialized inference pipeline.

In [ ]:
from src.models.manual_validation import run_manual_validation

# Run the manual validation script which tests 8 carefully crafted examples
run_manual_validation()

### Genuine Institutional Email Case Study
We tested a genuine legitimate institutional email:

*"Update on Salesforce Administrator Course Progress - 2028 Batch... Please note that the deadline to complete the Salesforce Administrator Course is 15th July 2026. It's compulsory..."*

The model incorrectly classified this email as phishing. This demonstrates a practical false-positive scenario: legitimate institutional emails naturally contain URLs, deadlines, forms, and urgency-like wording. Because the synthetic legitimate training data lacked these features, the model received a distorted view of reality.

## 8. Final Conclusions
All four evaluated classifiers achieved perfect performance on the provided dataset under the tested evaluation configurations. However, dataset auditing identified substantial template reuse and structural artifacts, while a small illustrative external manual validation set revealed significant generalization failures, particularly for legitimate emails containing URLs and urgency-like language. 

These findings demonstrate that high benchmark performance alone is insufficient evidence of robust real-world phishing detection. Future improvements should prioritize representative data collection, cross-dataset evaluation, and stronger external validation before considering production deployment.